# subgraphsearch quickstart

`subgraphsearch` finds **induced subgraphs with a given number of vertices and edges** (cliques, independent sets and everything in between), regular subgraphs, and subgraphs of a given total weight, by a fast randomized tabu search. This notebook walks through generating graphs, searching them, reading the results, and the visualizations.

In [ ]:
import subgraphsearch as ss
from subgraphsearch import viz

ss.__version__

## Graphs

Generate one of `ggen`'s random models, or read graph6, sparse6, Matrix Market, edge lists or `sub_search` adjacency lists. In a notebook a graph shows a summary, its degree distribution and, when small, a drawing.

In [ ]:
g = ss.generate("exponential", 100, 0.6, seed=1, rng="legacy")  # exactly ggen's `2 100 0 600 1 0 0 0`
g

## Reproducing the original program

`rng="legacy"` replays the C program's random numbers, so this finds the same five 11-cliques as the example in the original `sub_search` README, in the same order.

In [ ]:
cliques = ss.search(g, size=11, edges=55, experiments=5, max_moves=100, div_freq=25, div_dur=4, seed=72, rng="legacy")
cliques

## A harder search

The default generator (xoshiro256++) runs experiments in parallel. Here we look for 30 vertices spanning 343 edges, far denser than the graph's 58% average, and record the objective after every move (`trace=True`).

In [ ]:
r = ss.search(g, size=30, edges=343, experiments=60, max_moves=2000, div_freq=300, div_dur=8, seed=5, trace=True)
print(r.summary())

In [ ]:
viz.plot_search_summary(r);

The **run-time distribution** (left) is the probability of success within a given number of moves: still rising at the budget means more moves would help. The traces (right) show the bursts of random moves every `div_freq` moves.

At this density the target is barely attainable: every success found the *same* vertex set.

In [ ]:
len(r.solutions()), [len(ss.search(g, 30, m, experiments=60, max_moves=2000, div_freq=300, div_dur=8, seed=5).solutions()) for m in (330, 340)]

## Graph structure

Degree distribution against an Erdős–Rényi graph of the same density, the Laplacian spectrum (λ₁ > 0 means connected), and the adjacency matrix with the found subgraph moved to the top-left block.

In [ ]:
viz.plot_graph_summary(g, highlight=r.best.vertices);

`ggen`'s "power" model is not heavy-tailed, which the comparison makes plain:

In [ ]:
viz.plot_degree_distribution(ss.generate("power", 1500, 0.2, seed=52));

## Interactive view

A self-contained page: drag to pan, scroll to zoom, hover or click vertices, and pick a solution to highlight. `viz.save_html` writes the same page to a file.

In [ ]:
viz.show(ss.search(g, 30, 330, experiments=8, seed=2), max_conflict=0)

## Tuning

`ss.sweep` reruns a search across values of one parameter.

In [ ]:
sweep = ss.sweep(g, 30, 340, "div_dur", [0, 4, 16, 64], experiments=40, max_moves=1500, div_freq=300)
viz.plot_sweep(sweep, parameter="div_dur");

## Weighted graphs

The original `wggen` and `wsub_search` work with integer edge weights; the target is then a total weight. Here, 15 vertices whose edges weigh 180 in total, in a graph with a mean weight of 0.65 per vertex pair.

In [ ]:
w = ss.generate_weighted(200, 0.65, seed=1, rng="legacy")  # exactly wggen's `200 65 1`
rw = ss.search(w, 15, 180, experiments=40, max_moves=3000, div_freq=300, div_dur=10)
rw

## Regular subgraphs

Instead of an edge count, `degree=` asks for subgraphs in which every vertex has that many neighbors (the original `reg_search`), the search it was written for in strongly regular graphs. The Paley graph on 61 vertices is one; here we look for 4-regular subgraphs on 10 of its vertices. `two_phase=True` first finds the right number of edges (`ereg_search`).

In [ ]:
q = 61
squares = {x * x % q for x in range(1, q)}
paley = ss.Graph(q, [(i, j) for i in range(q) for j in range(i + 1, q) if (j - i) % q in squares])
reg = ss.search(paley, 10, degree=4, experiments=40, max_moves=2000, div_freq=300, div_dur=8)
print("degrees within the subgraph:", set(paley.induced(reg.best.vertices).degrees().tolist()))
reg

## Formats and other libraries

In [ ]:
import tempfile, pathlib
from subgraphsearch import convert

path = pathlib.Path(tempfile.mkdtemp()) / "g.g6"
g.write(path)
assert ss.read(path) == g
G = convert.to_networkx(g)
A = convert.to_scipy(g)
print(G, "|", repr(A))
print("formats:", ss.formats())

## The original interfaces

`ss.legacy` takes exactly what `sub_search` and `ggen` read on stdin and returns what they printed.

In [ ]:
print(ss.legacy.ggen("2 10 0 500 42 0 0 0"))